# Notebook 01: Exploração — Arquitetura GPU, Hierarquia de Memória e Modelo Roofline

## Aula 06: Aceleração com Hardware: GPU, TPU e Aceleradores

---

### Objetivos deste Notebook

1. Inspecionar hardware disponível (CPU/GPU) e suas capacidades
2. Compreender a hierarquia de memória GPU (registradores, SRAM, HBM)
3. Implementar e visualizar o **Modelo Roofline** (Williams, Waterman & Patterson, 2009)
4. Classificar operações de deep learning como compute-bound ou memory-bound
5. Benchmarkar operações fundamentais em diferentes precisões numéricas

### Conexão com o Material Teórico

Conforme discutido na seção **'Fundamentos da Arquitetura de GPUs para Machine Learning'**:

> "As GPUs NVIDIA contemporâneas são organizadas em torno de Streaming Multiprocessors (SMs), 
> unidades de processamento semi-independentes que coordenam a execução de milhares de threads 
> simultâneas." (Nickolls & Dally, 2010)

E na seção **'O Modelo Roofline: Caracterizando Workloads'**:

> $$\text{Performance}_{\text{atingível}} = \min\left(\text{Peak}_{\text{compute}},\ \text{BW}_{\text{mem}} \times \text{AI}\right)$$

---

In [ ]:
# Imports
import sys
import os
import time
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn

# Adicionar src/ ao path
sys.path.insert(0, os.path.abspath('..'))

from src.utils import get_device, get_hardware_info, set_seed, format_size
from src.data_preprocessing import DataPreprocessor
from src.evaluation import RooflineModel

# Configurações
logging.basicConfig(level=logging.INFO)
set_seed(42)
sns.set_theme(style='whitegrid')
%matplotlib inline

print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponível: {torch.cuda.is_available()}")

## 1. Inspeção de Hardware

Antes de otimizar, precisamos **conhecer o hardware**. Conforme discutido na seção 
'Fundamentos da Arquitetura de GPUs para Machine Learning':

- **Streaming Multiprocessors (SMs):** Unidades de processamento semi-independentes
- **Tensor Cores:** Hardware especializado para operações matriciais (Jia et al., 2018)
- **HBM (High Bandwidth Memory):** Memória de alta velocidade empilhada verticalmente

A arquitetura A100 disponibiliza 80GB de HBM2e com 2TB/s de largura de banda,
enquanto a H100 eleva para 80GB HBM3 com 3.35TB/s (NVIDIA, 2022).

In [ ]:
# Coleta informações detalhadas do hardware
# Implementa inspeção de capacidades relevantes: Tensor Cores, BF16, TF32
hw_info = get_hardware_info()

print("=" * 60)
print("INFORMAÇÕES DE HARDWARE")
print("=" * 60)
for key, value in hw_info.items():
    print(f"  {key:30s}: {value}")
print("=" * 60)

device = get_device()
print(f"\nDispositivo selecionado: {device}")

## 2. Formatos Numéricos: FP32, FP16, BF16, TF32

A seção **'Mixed Precision Training: Acelerando sem Comprometer'** (Micikevicius et al., 2018) detalha os formatos:

| Formato | Bits | Expoente | Mantissa | Range Dinâmico | Uso Principal |
|---------|------|----------|----------|----------------|---------------|
| FP32    | 32   | 8        | 23       | ~3.4×10³⁸      | Baseline / acumulação |
| FP16    | 16   | 5        | 10       | ~6.5×10⁴       | Forward/backward em Tensor Cores |
| BF16    | 16   | 8        | 7        | ~3.4×10³⁸      | Forward/backward (sem loss scaling) |
| TF32    | 19   | 8        | 10       | ~3.4×10³⁸      | Automático em Ampere+ |

> "BF16 mantém o mesmo range dinâmico do FP32, eliminando a necessidade de 
> loss scaling na maioria dos casos" (Kalamkar et al., 2019)

In [ ]:
# Demonstração visual dos formatos numéricos
# Ilustra diferenças de range e precisão discutidas na seção sobre Mixed Precision

formats_info = {
    'FP32': {'bits': 32, 'exponent': 8, 'mantissa': 23, 'max_val': 3.4e38, 'min_pos': 1.2e-38, 'torch_dtype': torch.float32},
    'FP16': {'bits': 16, 'exponent': 5, 'mantissa': 10, 'max_val': 6.55e4, 'min_pos': 6.1e-5, 'torch_dtype': torch.float16},
    'BF16': {'bits': 16, 'exponent': 8, 'mantissa': 7, 'max_val': 3.4e38, 'min_pos': 1.2e-38, 'torch_dtype': torch.bfloat16},
}

# Visualizar range de cada formato
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Range dinâmico (escala log)
names = list(formats_info.keys())
max_vals = [formats_info[n]['max_val'] for n in names]
colors = ['#2196F3', '#4CAF50', '#FF9800']

axes[0].barh(names, [np.log10(v) for v in max_vals], color=colors)
axes[0].set_xlabel('log₁₀(Valor Máximo)')
axes[0].set_title('Range Dinâmico por Formato Numérico')
for i, v in enumerate(max_vals):
    axes[0].text(np.log10(v) + 0.5, i, f'{v:.1e}', va='center', fontsize=10)

# Composição de bits
for i, name in enumerate(names):
    info = formats_info[name]
    # Sinal (1 bit) + expoente + mantissa
    axes[1].barh(i, 1, left=0, color='#F44336', height=0.5, label='Sinal' if i == 0 else '')
    axes[1].barh(i, info['exponent'], left=1, color='#2196F3', height=0.5, label='Expoente' if i == 0 else '')
    axes[1].barh(i, info['mantissa'], left=1 + info['exponent'], color='#4CAF50', height=0.5, label='Mantissa' if i == 0 else '')

axes[1].set_yticks(range(len(names)))
axes[1].set_yticklabels(names)
axes[1].set_xlabel('Número de Bits')
axes[1].set_title('Composição de Bits por Formato')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.savefig('../outputs/figures/formatos_numericos.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Figura salva em outputs/figures/formatos_numericos.png")

## 3. Geração e Exploração do Dataset

Geramos um dataset sintético de classificação de imagens para demonstrar as técnicas da aula.
O dataset é dimensionado para evidenciar diferenças de performance entre precisões numéricas.

In [ ]:
# Gerar dataset sintético
preprocessor = DataPreprocessor(n_samples=10000, img_size=32, n_channels=3, n_classes=10, seed=42)
df = preprocessor.generate_dataset()

print(f"Shape do dataset: {df.shape}")
print(f"\nPrimeiras 5 linhas:")
df.head()

In [ ]:
# Estatísticas do dataset
stats = preprocessor.get_dataset_stats(df)

print("\n=== ESTATÍSTICAS DO DATASET ===")
print(f"  Amostras:       {stats['n_samples']:,}")
print(f"  Features:       {stats['n_features']:,}")
print(f"  Classes:        {stats['n_classes']}")
print(f"  Média features: {stats['feature_mean']:.4f}")
print(f"  Std features:   {stats['feature_std']:.4f}")
print(f"  Memória:        {stats['memory_mb']:.1f} MB")

# Distribuição de classes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma de classes
class_dist = stats['class_distribution']
axes[0].bar(class_dist.keys(), class_dist.values(), color=sns.color_palette('tab10'))
axes[0].set_xlabel('Classe')
axes[0].set_ylabel('Frequência')
axes[0].set_title('Distribuição de Classes')

# Histograma de valores de features (amostra)
feature_sample = df.iloc[:, :100].values.flatten()
axes[1].hist(feature_sample, bins=50, alpha=0.7, color='#2196F3', edgecolor='white')
axes[1].set_xlabel('Valor do Pixel')
axes[1].set_ylabel('Frequência')
axes[1].set_title('Distribuição de Valores de Features (amostra)')

plt.tight_layout()
plt.savefig('../outputs/figures/dataset_exploration.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Modelo Roofline — Análise de Performance

O **Modelo Roofline** (Williams, Waterman & Patterson, 2009) oferece framework analítico 
para caracterizar workloads computacionais. A métrica central é a **intensidade aritmética**:

$$\text{Intensidade Aritmética} = \frac{\text{FLOPs}}{\text{Bytes Transferidos}}$$

Conforme discutido no material da aula:

> "Workloads com baixa intensidade aritmética são classificados como **memory-bound** — 
> sua performance é limitada pela velocidade de transferência de dados. Multiplicações 
> de matrizes densas exibem alta intensidade aritmética e são **compute-bound**."

### Tabela de Referência (do material da aula)

| Operação | Intensidade Aritmética | Classificação |
|----------|----------------------|---------------|
| Matmul (matrizes grandes) | ~100 FLOP/byte | Compute-bound |
| Convolução 3×3 | ~30 FLOP/byte | Balanceado |
| BatchNorm | ~2 FLOP/byte | Memory-bound |
| Softmax | ~1 FLOP/byte | Memory-bound |
| LayerNorm | ~3 FLOP/byte | Memory-bound |

In [ ]:
# Implementação do Modelo Roofline — Williams, Waterman & Patterson (2009)
# Compara perfis de hardware de diferentes GPUs

print("=== PERFIS DE HARDWARE DISPONÍVEIS ===")
for name, profile in RooflineModel.HARDWARE_PROFILES.items():
    roofline = RooflineModel.from_hardware(name)
    print(f"  {name:15s}: Peak={profile['peak_tflops']:>7.0f} TFLOPS, "
          f"BW={profile['bw_gb_s']:>6.0f} GB/s, "
          f"Ridge Point={roofline.ridge_point:.1f} FLOP/byte")

In [ ]:
# Análise de operações usando Modelo Roofline para A100
# Classifica operações de DL como compute-bound ou memory-bound
# Conforme Tabela 1 e Figura 1 do material da aula (Dao et al., 2022)

roofline_a100 = RooflineModel.from_hardware('A100_80GB')
analysis = roofline_a100.analyze_operations()

print(f"\n=== ANÁLISE ROOFLINE — NVIDIA A100 ===")
print(f"  Peak Compute: {roofline_a100.peak_compute_tflops} TFLOPS")
print(f"  Bandwidth:    {roofline_a100.memory_bandwidth_gb_s} GB/s")
print(f"  Ridge Point:  {roofline_a100.ridge_point:.1f} FLOP/byte\n")

for op in analysis:
    status = '🟢' if op['classification'] == 'compute-bound' else '🔴' if op['classification'] == 'memory-bound' else '🟡'
    print(f"  {status} {op['operation']:20s} | AI={op['arithmetic_intensity']:>6.1f} FLOP/byte | "
          f"Perf={op['attainable_tflops']:>7.1f} TFLOPS | {op['classification']:15s} | "
          f"Utilização={op['utilization_pct']:.1f}%")

In [ ]:
# Visualização do Modelo Roofline — Figura 1 do material da aula
# "Representação do Modelo Roofline mostrando regiões memory-bound e compute-bound"

from src.evaluation import plot_roofline

fig = plot_roofline(
    roofline_a100,
    operations=RooflineModel.TYPICAL_AI,
    save_path='../outputs/figures/roofline_a100.png',
    title='Modelo Roofline — NVIDIA A100 80GB'
)

# Recriar para display (plot_roofline fecha a figura)
data = roofline_a100.get_roofline_data()
fig, ax = plt.subplots(figsize=(10, 6))
ax.loglog(data['arithmetic_intensity'], data['attainable_performance'], 'b-', linewidth=2, label='Roofline')
ax.axvline(data['ridge_point'], color='gray', linestyle='--', alpha=0.5, label=f"Ridge Point ({data['ridge_point']:.1f} FLOP/byte)")

colors_map = {'compute-bound': 'green', 'memory-bound': 'red', 'balanced': 'orange'}
for op_name, ai in RooflineModel.TYPICAL_AI.items():
    perf = roofline_a100.attainable_performance(ai)
    cls = roofline_a100.classify_operation(ai)
    ax.plot(ai, perf, 'o', color=colors_map[cls], markersize=10)
    ax.annotate(op_name.replace('_', ' '), (ai, perf), textcoords='offset points', xytext=(5, 8), fontsize=9)

ax.set_xlabel('Intensidade Aritmética (FLOP/byte)', fontsize=12)
ax.set_ylabel('Performance Atingível (TFLOPS)', fontsize=12)
ax.set_title('Modelo Roofline — NVIDIA A100 80GB\n(Williams, Waterman & Patterson, 2009)', fontsize=13)
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3, which='both')

# Anotar regiões
ax.text(0.3, 50, 'MEMORY-BOUND', fontsize=12, color='red', alpha=0.5, fontweight='bold')
ax.text(200, 50, 'COMPUTE-BOUND', fontsize=12, color='green', alpha=0.5, fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Benchmark de Operações Fundamentais

Vamos medir empiricamente o tempo de operações para validar as classificações do modelo roofline.

Conforme o material da aula:
> "Estudos de profiling em Vision Transformers indicam que aproximadamente 40% do tempo 
> de execução concentra-se em operações memory-bound, apesar de representarem fração menor 
> do total de FLOPs" (Dao et al., 2022)

In [ ]:
# Benchmark de operações fundamentais — valida classificação do roofline
# Compara operações compute-bound vs memory-bound

def benchmark_operation(op_fn, x, n_iter=100, warmup=20, device_type='cpu'):
    """Benchmarka uma operação."""
    # Warmup
    for _ in range(warmup):
        _ = op_fn(x)
    if device_type == 'cuda':
        torch.cuda.synchronize()
    
    times = []
    for _ in range(n_iter):
        if device_type == 'cuda':
            torch.cuda.synchronize()
        start = time.perf_counter()
        _ = op_fn(x)
        if device_type == 'cuda':
            torch.cuda.synchronize()
        times.append(time.perf_counter() - start)
    return np.mean(times) * 1000, np.std(times) * 1000  # ms

# Configuração
B, D = 64, 512  # Batch size e dimensão (múltiplos de 64 para Tensor Cores)
x_bench = torch.randn(B, D, device=device)

# Definir operações
matmul_weight = torch.randn(D, D, device=device)
layernorm = nn.LayerNorm(D).to(device)
batchnorm = nn.BatchNorm1d(D).to(device).eval()

operations = {
    'MatMul (DxD)': lambda x: x @ matmul_weight,
    'LayerNorm': lambda x: layernorm(x),
    'BatchNorm': lambda x: batchnorm(x),
    'Softmax': lambda x: torch.softmax(x, dim=-1),
    'GELU': lambda x: torch.nn.functional.gelu(x),
    'Element-wise Add': lambda x: x + x,
}

print(f"\n=== BENCHMARK DE OPERAÇÕES ({device}) ===")
print(f"  Tensor shape: ({B}, {D})")
print(f"{'Operação':25s} {'Tempo (ms)':>12s} {'Std (ms)':>10s} {'Tipo':>15s}")
print("-" * 65)

bench_results = {}
for name, op_fn in operations.items():
    avg_ms, std_ms = benchmark_operation(op_fn, x_bench, device_type=device.type)
    classification = 'compute-bound' if 'MatMul' in name else 'memory-bound'
    bench_results[name] = {'avg_ms': avg_ms, 'std_ms': std_ms, 'type': classification}
    print(f"  {name:23s} {avg_ms:10.4f} ms  {std_ms:8.4f} ms  {classification:>15s}")

In [ ]:
# Visualização do benchmark de operações
fig, ax = plt.subplots(figsize=(10, 5))

op_names = list(bench_results.keys())
op_times = [bench_results[n]['avg_ms'] for n in op_names]
op_colors = ['#4CAF50' if bench_results[n]['type'] == 'compute-bound' else '#F44336' for n in op_names]

bars = ax.barh(op_names, op_times, color=op_colors)
ax.set_xlabel('Tempo Médio (ms)', fontsize=12)
ax.set_title(f'Benchmark de Operações DL — {device}', fontsize=13)

# Legenda manual
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor='#4CAF50', label='Compute-bound'),
    Patch(facecolor='#F44336', label='Memory-bound'),
], loc='lower right')

for bar, t in zip(bars, op_times):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{t:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/figures/benchmark_operacoes.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Comparação de Aceleradores

Conforme **Tabela 2** do material da aula:

| Acelerador | Força | Limitação | Caso de Uso Ideal |
|------------|-------|-----------|-------------------|
| GPU (NVIDIA) | Flexibilidade, ecossistema | Custo energético | Treinamento, pesquisa |
| TPU (Google) | Matmul massivo, pods | Ecossistema limitado | Treinamento de LLMs |
| NPU (edge) | Eficiência energética | Capacidade limitada | Inferência mobile |
| AWS Inferentia | Custo-benefício | Apenas inferência | Produção em escala |

> "TPUs foram projetadas desde o início exclusivamente para acelerar operações tensoriais" (Jouppi et al., 2017)

In [ ]:
# Comparação visual de aceleradores — Tabela 2 do material
# Dados baseados em especificações públicas (NVIDIA, Google, AWS)

accelerators = {
    'NVIDIA A100': {'tflops_fp16': 312, 'bw_gb_s': 2039, 'tdp_w': 400, 'use': 'Treinamento'},
    'NVIDIA H100': {'tflops_fp16': 989, 'bw_gb_s': 3352, 'tdp_w': 700, 'use': 'Treinamento'},
    'Google TPU v4': {'tflops_fp16': 275, 'bw_gb_s': 1200, 'tdp_w': 170, 'use': 'Treinamento LLMs'},
    'NVIDIA T4': {'tflops_fp16': 65, 'bw_gb_s': 300, 'tdp_w': 70, 'use': 'Inferência'},
    'AWS Inferentia': {'tflops_fp16': 128, 'bw_gb_s': 300, 'tdp_w': 75, 'use': 'Inferência'},
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

names = list(accelerators.keys())
accel_colors = ['#1976D2', '#0D47A1', '#E65100', '#43A047', '#7B1FA2']

# TFLOPS
tflops = [accelerators[n]['tflops_fp16'] for n in names]
axes[0].barh(names, tflops, color=accel_colors)
axes[0].set_xlabel('TFLOPS (FP16)')
axes[0].set_title('Performance de Pico')

# Bandwidth
bws = [accelerators[n]['bw_gb_s'] for n in names]
axes[1].barh(names, bws, color=accel_colors)
axes[1].set_xlabel('Memory BW (GB/s)')
axes[1].set_title('Largura de Banda de Memória')

# Eficiência (TFLOPS/Watt)
efficiency = [accelerators[n]['tflops_fp16'] / accelerators[n]['tdp_w'] for n in names]
axes[2].barh(names, efficiency, color=accel_colors)
axes[2].set_xlabel('TFLOPS/Watt')
axes[2].set_title('Eficiência Energética')

plt.suptitle('Comparação de Aceleradores para Machine Learning\n(Jouppi et al., 2017; NVIDIA, 2022)', 
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../outputs/figures/comparacao_aceleradores.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Preparação de Dados para Treinamento

Preparamos os DataLoaders com `pin_memory=True` para transferência assíncrona CPU→GPU e
batch sizes múltiplos de 8, conforme recomendações de alinhamento com Tensor Cores.

In [ ]:
# Preparar dados para os notebooks seguintes
df = preprocessor.load_data()
df = preprocessor.clean_data(df)
features, labels = preprocessor.prepare_features(df)

# Dividir em treino/validação/teste
splits = preprocessor.split_data(features, labels)

for split_name, (X, y) in splits.items():
    print(f"  {split_name:6s}: {X.shape[0]:>5,} amostras, shape features={X.shape}, n_classes={len(np.unique(y))}")

# Criar DataLoaders — batch_size=64 (múltiplo de 8 para Tensor Cores)
loaders = preprocessor.create_dataloaders(splits, batch_size=64, pin_memory=True)

# Verificar um batch
sample_batch, sample_labels = next(iter(loaders['train']))
print(f"\nBatch shape: {sample_batch.shape}  (B, C, H, W)")
print(f"Labels shape: {sample_labels.shape}")
print(f"Dtype: {sample_batch.dtype}")

## Resumo

Neste notebook, exploramos:

1. **Hardware disponível** — capacidades de GPU, Tensor Cores, suporte a BF16/TF32
2. **Formatos numéricos** — FP32, FP16, BF16 e seus trade-offs (Micikevicius et al., 2018)
3. **Modelo Roofline** — classificação de operações como compute/memory-bound (Williams et al., 2009)
4. **Benchmark de operações** — validação empírica das classificações teóricas
5. **Comparação de aceleradores** — GPU, TPU, NPU, Inferentia (Jouppi et al., 2017)

---

**Próximo notebook:** [02_treinamento.ipynb](02_treinamento.ipynb) — Treinamento com Mixed Precision e Gradient Checkpointing